### Setting

In [ ]:
# Device

import torch

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("GPU is available and will be used.")
else:
    device = torch.device("cpu")
    print("No GPU available, using CPU.")

In [ ]:
#Imports

import os
import json
import random
import pandas as pd
import numpy as np
from transformers import (
    BertTokenizer, BertConfig, BertForMaskedLM,
    TapasTokenizer, TapasForMaskedLM,
)
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from dataset import create_data

from utils import evaluate_masked_prediction

In [ ]:
# Tokenizer & config

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
config = BertConfig.from_pretrained('bert-base-uncased')

### Models

In [ ]:
# BERT
bert_base = BertForMaskedLM.from_pretrained('bert-base-uncased')
bert_base = bert_base.to(device)

# TaPas
tapas_name = "google/tapas-base-masklm"
tapas_tokenizer = TapasTokenizer.from_pretrained(tapas_name)
tapas = TapasForMaskedLM.from_pretrained(tapas_name)
tapas.to(device)

In [ ]:
from model import HAETAE
from no_cl import HAETAE_INTERPOLATE
from no_ip import HAETAE_NEWLOSS

In [ ]:
'''
ours_path_movie = '/workspace/models/movie_complete/epoch-9'
no_cl_path_movie = '/workspace/models/movie_no_cl/epoch-9'
no_ip_path_movie = '/workspace/models/movie_no_ip/epoch-9'
bert_path_movie = '/workspace/models/movie_bert/epoch-9'
'''

ours_movie = HAETAE(config, tokenizer, ours_path_movie)
ours_movie = ours_movie.to(device)

no_cl_movie = HAETAE_INTERPOLATE(config, tokenizer, no_cl_path_movie)
no_cl_movie = no_cl_movie.to(device)

no_ip_movie = HAETAE_NEWLOSS(config, tokenizer, no_ip_path_movie)
no_ip_movie = no_ip_movie.to(device)

bert_movie = BertForMaskedLM.from_pretrained(bert_path_movie, local_files_only=True)
bert_movie = bert_movie.to(device)

In [ ]:
'''
ours_path_product = '/workspace/models/product_complete/epoch-9'
no_cl_path_product = '/workspace/models/product_no_cl/epoch-9'
no_ip_path_product = '/workspace/models/product_no_ip/epoch-9'
bert_path_product = '/workspace/models/product_bert/epoch-9'
'''

ours_product = HAETAE(config, tokenizer, ours_path_product)
ours_product = ours_product.to(device)

no_cl_product = HAETAE_INTERPOLATE(config, tokenizer, no_cl_path_product)
no_cl_product = no_cl_product.to(device)

no_ip_product = HAETAE_NEWLOSS(config, tokenizer, no_ip_path_product)
no_ip_product = no_ip_product.to(device)

bert_product = BertForMaskedLM.from_pretrained(bert_path_product, local_files_only=True)
bert_product = bert_product.to(device)

### Data

In [ ]:
product = create_data("./data/product_test.jsonl", path_is="json")
movie = create_data("./data/movie_test.jsonl", path_is="json")

In [ ]:
movie[0]

### Masked Prediction

In [ ]:
# In-domain: Movie

# Pre-trained: BERT, TaPas, TaBERT
print("BERT's performance in Masked Prediction")
evaluate_masked_prediction(movie, 'Key', bert_base, tokenizer)
evaluate_masked_prediction(movie, 'Value', bert_base, tokenizer)
print("\n")

print("TaPas's performance in Masked Prediction")
evaluate_masked_prediction(movie, 'Key', tapas, tapas_tokenizer)
evaluate_masked_prediction(movie, 'Value', tapas, tapas_tokenizer)
print("\n")

# Domain-specific pre-trained: Ours, No CL, No IP, trained BERT

print("Ours's performance in Masked Prediction")
evaluate_masked_prediction(movie, 'Key', ours_movie, tokenizer)
evaluate_masked_prediction(movie, 'Value', ours_movie, tokenizer)
print("\n")

print("Masked Prediction performance without Regularization")
evaluate_masked_prediction(movie, 'Key', no_cl_movie, tokenizer)
evaluate_masked_prediction(movie, 'Value', no_cl_movie, tokenizer)
print("\n")

print("Masked Prediction performance without Interpolation")
evaluate_masked_prediction(movie, 'Key', no_ip_movie, tokenizer)
evaluate_masked_prediction(movie, 'Value', no_ip_movie, tokenizer)
print("\n")

print("Domain-specific pre-trained BERT's performance in Masked Prediction")
evaluate_masked_prediction(movie, 'Key', bert_movie, tokenizer)
evaluate_masked_prediction(movie, 'Value', bert_movie, tokenizer)
print("\n")

In [ ]:
# In-domain: product

# Pre-trained: BERT, TaPas, TaBERT
print("BERT's performance in Masked Prediction")
evaluate_masked_prediction(product, 'Key', bert_base, tokenizer)
evaluate_masked_prediction(product, 'Value', bert_base, tokenizer)
print("\n")

print("TaPas's performance in Masked Prediction")
evaluate_masked_prediction(product, 'Key', tapas, tapas_tokenizer)
evaluate_masked_prediction(product, 'Value', tapas, tapas_tokenizer)
print("\n")

# Domain-specific pre-trained: Ours, No CL, No IP, trained BERT

print("Ours's performance in Masked Prediction")
evaluate_masked_prediction(product, 'Key', ours_product, tokenizer)
evaluate_masked_prediction(product, 'Value', ours_product, tokenizer)
print("\n")

print("Masked Prediction performance without Regularization")
evaluate_masked_prediction(product, 'Key', no_cl_product, tokenizer)
evaluate_masked_prediction(product, 'Value', no_cl_product, tokenizer)
print("\n")

print("Masked Prediction performance without Interpolation")
evaluate_masked_prediction(product, 'Key', no_ip_product, tokenizer)
evaluate_masked_prediction(product, 'Value', no_ip_product, tokenizer)
print("\n")

print("Domain-specific pre-trained BERT's performance in Masked Prediction")
evaluate_masked_prediction(product, 'Key', bert_product, tokenizer)
evaluate_masked_prediction(product, 'Value', bert_product, tokenizer)
print("\n")

In [ ]:
# Cross-domain

# Trained on Product -> Tested on Movie
print("Ours's performance in Cross-domain Masked Prediction")
evaluate_masked_prediction(movie, 'Key', ours_product, tokenizer)
evaluate_masked_prediction(movie, 'Value', ours_product, tokenizer)
print("\n")

print("BERT's performance in Cross-domain Masked Prediction")
evaluate_masked_prediction(movie, 'Key', bert_product, tokenizer)
evaluate_masked_prediction(movie, 'Value', bert_product, tokenizer)
print("\n")


# Trained on Movie -> Tested on Product
print("Ours's performance in Cross-domain Masked Prediction")
evaluate_masked_prediction(product, 'Key', ours_movie, tokenizer)
evaluate_masked_prediction(product, 'Value', ours_movie, tokenizer)
print("\n")

print("BERT's performance in Cross-domain Masked Prediction")
evaluate_masked_prediction(product, 'Key', bert_movie, tokenizer)
evaluate_masked_prediction(product, 'Value', bert_movie, tokenizer)
print("\n")